# Passo 1 — differenze di stato medio SENS − CTRL (calcolo)

Prima di chiedersi se la vegetazione osservata migliora la previsione, si
verifica se e come cambia il clima del modello. In SENS LAI, cover e tipi di
vegetazione sono osservati e prescritti; in CTRL vengono da un run storico di
LPJ-GUESS. La differenza è soprattutto di **stato medio**, ed è quella che si
guarda qui (le regressioni di `cover_regression/` tolgono la media e lavorano
solo sulle anomalie interannuali, la parte più piccola).

Per ogni stagione e lead singolo (0–4):

- **variabili del modello** (`tas`, `alb`, `snd`): media d'ensemble per anno di
  SENS e CTRL, differenza per anno, media nel tempo e p-value col bootstrap a
  blocchi del repo (lo stesso delle mappe di bias, Fig. 4);
- **vegetazione prescritta** (`cvh`, `cvl` efficaci, `lai_hv`, `lai_lv`): serie
  stagionali per anno di SENS e CTRL, con la stessa etichetta degli anni dei
  file del modello (anno del primo mese della stagione, dicembre per DJF).

Ingresso: i file stagionali per lead singolo prodotti da
`03-decadal_season_aggregation` (`WORK_DIR/<exp>/..._lead<k>_rad_<stagione>.nc`).
Uscita: `.nc` in `WORK_DIR/cover_mechanism/`. Figure: `01-Fig_mean_state_delta`.

In [ ]:
# rende config.py (in notebooks/) importabile anche da questa sottocartella
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import WORK_DIR, FIG_DIR

exp_ctrl = 'a1ua'
exp_sens = 'a52o'
MODEL_VARS = ['tas', 'alb', 'snd']
COVER_VARS = ['cvh', 'cvl', 'lai_hv', 'lai_lv']
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']
LEADS = [0, 1, 2, 3, 4]
N_ITER = 1000        # iterazioni del bootstrap a blocchi (come le mappe di bias)
N_PROC = 2           # processi in parallelo

OUT_DIR = WORK_DIR / 'cover_mechanism'
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Le funzioni stanno in mechanism_lib.py: i processi spawn usati sotto non
# vedono le funzioni definite nel notebook.
sys.path.insert(0, os.getcwd())
import mechanism_lib as ml

# Controllo degli ingressi: numero di file per membro per ogni lead (0 = mancano)
ml.check_inputs(exp_ctrl, exp_sens, MODEL_VARS, COVER_VARS, SEASONS, LEADS);

Se una variabile ha 0 file per qualche stagione o lead, va prima prodotta con
`03-decadal_season_aggregation` (negli originali, `00-process_2025/`),
impostando `exp` e `var` (per `alb` e `snd` la versione stagionale per lead
singolo potrebbe non essere mai stata lanciata). Le combinazioni mancanti
vengono saltate con un messaggio di errore e il resto procede.

In [ ]:
%%time
import multiprocessing as mp

# Vegetazione: serie stagionali per anno (veloce)
jobs = [(exp_ctrl, exp_sens, var, season, OUT_DIR) for var in COVER_VARS for season in SEASONS]
with mp.get_context('spawn').Pool(processes=N_PROC, maxtasksperchild=1) as pool:
    for _i, msg in enumerate(pool.imap_unordered(ml.run_one_cover, jobs), 1):
        print(f"  {msg}   {_i}/{len(jobs)}", flush=True)

In [ ]:
%%time
# Variabili del modello: differenze per anno, media e p-value (lento: bootstrap
# a blocchi con N_ITER iterazioni per ogni variabile, stagione e lead)
jobs = [(exp_ctrl, exp_sens, var, season, lead, OUT_DIR, N_ITER)
        for var in MODEL_VARS for season in SEASONS for lead in LEADS]
with mp.get_context('spawn').Pool(processes=N_PROC, maxtasksperchild=1) as pool:
    for _i, msg in enumerate(pool.imap_unordered(ml.run_one_model, jobs), 1):
        print(f"  {msg}   {_i}/{len(jobs)}", flush=True)